In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

path = "/content/drive/MyDrive"

for root, dirs, files in os.walk(path):
    for file in files:
        if "HI" in file:
            print(os.path.join(root, file))

/content/drive/MyDrive/Resume WINNER CHIOMA UNUAGBON 2.pdf
/content/drive/MyDrive/Resume WINNER CHIOMA UNUAGBON 4.pdf
/content/drive/MyDrive/Birth Certificate - WINNER CHIOMA UNUAGBON.pdf
/content/drive/MyDrive/HI-Small_Trans.csv
/content/drive/MyDrive/HI SMALL PATTERNS.xlsx
/content/drive/MyDrive/Colab Notebooks/EDA NOTEBOOK_WINNER CHIOMA UNUAGBON.ipynb


In [ ]:
df = transactions.copy()

In [ ]:
transactions.head()

,timestamp,sender_bank,sender_account,receiver_bank,receiver_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [ ]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Data columns (total 11 columns):
 #   Column              Dtype         
---  ------              -----         
 0   timestamp           datetime64[ns]
 1   sender_bank         int64         
 2   sender_account      object        
 3   receiver_bank       int64         
 4   receiver_account    object        
 5   amount_received     float64       
 6   receiving_currency  object        
 7   amount_paid         float64       
 8   payment_currency    object        
 9   payment_format      object        
 10  is_laundering       int64         
dtypes: datetime64[ns](1), float64(2), int64(3), object(5)
memory usage: 426.2+ MB


In [ ]:
transactions.isnull().sum()

,0
timestamp,0
sender_bank,0
sender_account,0
receiver_bank,0
receiver_account,0
amount_received,0
receiving_currency,0
amount_paid,0
payment_currency,0
payment_format,0


In [ ]:
patterns = pd.read_excel(
    "/content/drive/MyDrive/HI SMALL PATTERNS.xlsx",
    engine="openpyxl"
)

In [ ]:
patterns.head(5)

,BEGIN LAUNDERING ATTEMPT - FAN-OUT: Max 16-degree Fan-Out
0,"2022/09/01 00:06,021174,800737690,012,80011F99..."
1,"2022/09/01 04:33,021174,800737690,020,80020C5B..."
2,"2022/09/01 09:14,021174,800737690,020,80006A5E..."
3,"2022/09/01 09:56,021174,800737690,00220,8007A5..."
4,"2022/09/01 11:28,021174,800737690,001244,80093..."


In [ ]:
transactions["is_laundering"].value_counts()

,count
is_laundering,
0,5073168
1,5177


In [ ]:
df["is_laundering"].value_counts(normalize=True)

,proportion
is_laundering,
0,0.998981
1,0.001019


In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [ ]:
aml_df = df.groupby(
    ["sender_bank", "sender_account"]
).agg(
    transaction_count=("sender_account", "size"),
    total_sent=("amount_paid", "sum"),
    unique_receivers=("receiver_account", "nunique"),
    first_txn=("timestamp", "min"),
    last_txn=("timestamp", "max"),
    laundering_count=("is_laundering", "sum"),
    laundering_rate=("is_laundering", "mean")
).reset_index()

In [ ]:
received = (
    df.groupby("receiver_account")["amount_received"]
      .sum()
      .reset_index(name="total_received")
)

aml_df = aml_df.merge(
    received,
    left_on="sender_account",
    right_on="receiver_account",
    how="left"
)

aml_df["total_received"] = aml_df["total_received"].fillna(0)

aml_df.drop(columns=["receiver_account"], inplace=True)

In [ ]:
fan_out = (
    df.groupby("sender_account")["receiver_account"]
      .nunique()
      .reset_index(name="fan_out")
)

aml_df = aml_df.merge(
    fan_out,
    on="sender_account",
    how="left"
)

In [ ]:
aml_df["fan_out_percentile"] = aml_df["fan_out"].rank(pct=True)

In [ ]:
fan_in = (
    df.groupby("receiver_account")["sender_account"]
      .nunique()
      .reset_index(name="fan_in")
)

aml_df = aml_df.merge(
    fan_in,
    left_on="sender_account",
    right_on="receiver_account",
    how="left"
)

aml_df["fan_in"] = aml_df["fan_in"].fillna(0)

aml_df.drop(columns=["receiver_account"], inplace=True)

In [ ]:
aml_df["fan_in_percentile"] = aml_df["fan_in"].rank(pct=True)

In [ ]:
aml_df["active_days"] = (
    aml_df["last_txn"] -
    aml_df["first_txn"]
).dt.days + 1

In [ ]:
aml_df["txn_percentile"] = aml_df["transaction_count"].rank(pct=True)

In [ ]:
aml_df["sent_percentile"] = aml_df["total_sent"].rank(pct=True)

In [ ]:
aml_df["risk_score"] = (
      35 * aml_df["txn_percentile"]
    + 25 * aml_df["fan_out_percentile"]
    + 15 * aml_df["fan_in_percentile"]
    + 15 * aml_df["sent_percentile"]
    + 10 * aml_df["laundering_rate"]
)

In [ ]:
aml_df["risk_category"] = pd.qcut(
    aml_df["risk_score"],
    q=[0, 0.80, 0.95, 0.99, 1],
    labels=["Low", "Medium", "High", "Critical"]
)

In [ ]:
def risk_driver(row):

    drivers = []

    if row["txn_percentile"] >= 0.95:
        drivers.append("High Transaction Activity")

    if row["fan_out_percentile"] >= 0.95:
        drivers.append("High Fan-Out Activity")

    if row["fan_in_percentile"] >= 0.95:
        drivers.append("High Fan-In Activity")

    if row["sent_percentile"] >= 0.95:
        drivers.append("High Value Movement")

    if row["laundering_count"] > 0:
        drivers.append("Known Laundering Activity")

    if len(drivers) == 0:
        return "Normal Activity"

    return ", ".join(drivers)

aml_df["risk_driver"] = aml_df.apply(risk_driver, axis=1)

In [ ]:
aml_df["risk_driver"] = aml_df.apply(
    risk_driver,
    axis=1
)

In [ ]:
aml_df.drop(columns=["sent_norm"], inplace=True)

In [ ]:
print(aml_df.columns.to_list())

['sender_bank', 'sender_account', 'transaction_count', 'total_sent', 'unique_receivers', 'first_txn', 'last_txn', 'laundering_count', 'laundering_rate', 'total_received', 'fan_out', 'fan_in', 'active_days', 'txn_percentile', 'fan_out_percentile', 'fan_in_percentile', 'risk_score', 'risk_category', 'sent_percentile', 'risk_driver']


In [ ]:
aml_table = aml_df[[





    "sender_bank",

    "sender_account",



    "transaction_count",

    "txn_percentile",

    "first_txn",

    "last_txn",

    "active_days",



    "total_sent",

    "total_received",



    "unique_receivers",

    "fan_out",

    "fan_out_percentile",

    "fan_in",

    "fan_in_percentile",



    "laundering_count",

    "laundering_rate",



    "risk_score",

    "risk_category",

    "risk_driver"

]].copy()

In [ ]:
aml_df.head(10)

,sender_bank,sender_account,transaction_count,total_sent,unique_receivers,first_txn,last_txn,laundering_count,laundering_rate,total_received,fan_out,fan_in,active_days,txn_percentile,fan_out_percentile,fan_in_percentile,risk_score,risk_category,sent_percentile,risk_driver
0,1,800042CB0,134,1.275401e+06,13,2022-09-01 00:13:00,2022-09-10 15:20:00,0,0.000000,40018.84,13,1.0,10,0.997579,0.996796,0.346126,78.194866,Medium,0.877853,"High Transaction Activity, High Fan-Out Activity"
1,1,800054E60,222,2.781327e+07,13,2022-09-01 02:47:00,2022-09-10 18:55:00,0,0.000000,975938.43,13,2.0,10,0.999860,0.996796,0.584992,83.260857,High,0.971398,"High Transaction Activity, High Fan-Out Activi..."
2,1,8000555D0,128,1.596319e+08,9,2022-09-01 00:20:00,2022-09-10 22:14:00,0,0.000000,3158455.22,9,2.0,10,0.997034,0.984577,0.584992,83.121299,High,0.989052,"High Transaction Activity, High Fan-Out Activi..."
3,1,800056160,89,2.106860e+05,11,2022-09-01 00:02:00,2022-09-10 17:20:00,1,0.011236,6227.20,11,7.0,10,0.988884,0.993173,0.987852,85.800198,Critical,0.761985,"High Transaction Activity, High Fan-Out Activi..."
4,1,800056370,131,5.500107e+07,9,2022-09-01 00:04:00,2022-09-10 23:42:00,0,0.000000,81579.64,9,2.0,10,0.997288,0.984577,0.584992,82.994055,High,0.979978,"High Transaction Activity, High Fan-Out Activi..."
5,1,800056A50,107,1.833335e+07,13,2022-09-01 02:03:00,2022-09-10 17:57:00,0,0.000000,49590.67,13,1.0,10,0.994073,0.996796,0.346126,79.363545,Medium,0.963946,"High Transaction Activity, High Fan-Out Activi..."
6,1,800056CB0,114,1.404377e+06,12,2022-09-01 00:02:00,2022-09-10 07:56:00,0,0.000000,40942.76,12,1.0,10,0.995312,0.995318,0.346126,78.148995,Medium,0.882549,"High Transaction Activity, High Fan-Out Activity"
7,1,800057020,210,3.497914e+06,15,2022-09-01 01:46:00,2022-09-10 23:21:00,0,0.000000,142707.98,15,1.0,10,0.999826,0.998359,0.346126,78.957337,Medium,0.920837,"High Transaction Activity, High Fan-Out Activity"
8,1,800057620,45,2.006922e+06,3,2022-09-01 00:03:00,2022-09-09 22:16:00,0,0.000000,98948.11,3,2.0,9,0.947933,0.848399,0.584992,76.647951,Medium,0.899028,Normal Activity
9,1,800057A10,123,4.913023e+07,6,2022-09-01 00:00:00,2022-09-10 22:29:00,0,0.000000,49099916.06,6,1.0,10,0.996508,0.956161,0.346126,78.653569,Medium,0.978658,"High Transaction Activity, High Fan-Out Activi..."


In [ ]:
aml_df.tail(5)

,sender_bank,sender_account,transaction_count,total_sent,unique_receivers,first_txn,last_txn,laundering_count,laundering_rate,total_received,fan_out,fan_in,active_days,txn_percentile,fan_out_percentile,fan_in_percentile,risk_score,risk_category,sent_percentile,risk_driver
496994,356295,81495EE41,2,0.008984,1,2022-09-02 08:39:00,2022-09-09 17:12:00,0,0.0,0.0,1,0.0,8,0.466561,0.28894,0.095019,25.012153,Low,0.002247,Normal Activity
496995,356296,81495F061,2,0.018728,1,2022-09-02 03:02:00,2022-09-09 20:04:00,0,0.0,0.0,1,0.0,8,0.466561,0.28894,0.095019,25.024346,Low,0.003060,Normal Activity
496996,356300,814960F41,3,0.584486,1,2022-09-01 10:39:00,2022-09-09 21:24:00,0,0.0,0.0,1,0.0,9,0.669720,0.28894,0.095019,32.264004,Low,0.011668,Normal Activity
496997,356302,814962711,2,0.009458,1,2022-09-02 14:16:00,2022-09-09 18:54:00,0,0.0,0.0,1,0.0,8,0.466561,0.28894,0.095019,25.012802,Low,0.002291,Normal Activity
496998,356303,814963711,2,0.020512,1,2022-09-02 09:23:00,2022-09-09 11:01:00,0,0.0,0.0,1,0.0,8,0.466561,0.28894,0.095019,25.026489,Low,0.003203,Normal Activity


In [ ]:
aml_df["risk_score"].describe()

,risk_score
count,496999.000000
mean,45.017917
std,19.978070
min,14.027563
25%,27.160507
50%,40.109145
75%,62.832179
max,92.017449


In [ ]:
aml_df["risk_category"].value_counts()

,count
risk_category,
Low,397599
Medium,74550
High,19880
Critical,4970


In [ ]:
aml_df["transaction_count"].describe()

,transaction_count
count,496999.000000
mean,10.218019
std,290.282946
min,1.000000
25%,1.000000
50%,2.000000
75%,5.000000
max,168672.000000


In [ ]:
aml_df["txn_percentile"].describe()

,txn_percentile
count,496999.000000
mean,0.500001
std,0.279578
min,0.153677
25%,0.153677
50%,0.466561
75%,0.747544
max,1.000000


In [ ]:
aml_df.shape

(496999, 20)

In [ ]:
def risk_driver(row):

    if row["laundering_count"] > 0:
        return "Laundering"

    elif row["txn_percentile"] >= 0.95:
        return "High Activity"

    elif row["fan_out_percentile"] >= 0.95:
        return "Fan-Out"

    elif row["fan_in_percentile"] >= 0.95:
        return "Fan-In"

    elif row["sent_percentile"] >= 0.95:
        return "High Value"

    else:
        return "Normal"

aml_df["risk_driver"] = aml_df.apply(risk_driver, axis=1)

In [ ]:
aml_df["risk_driver"].value_counts().head(10)

,count
risk_driver,
Normal,418484
High Activity,24143
Fan-In,23001
High Value,15444
Fan-Out,12551
Laundering,3376


In [ ]:
aml_df["risk_category"].value_counts()

,count
risk_category,
Low,397599
Medium,74550
High,19880
Critical,4970


In [ ]:
import numpy as np

aml_df["txn_log"] = np.log1p(aml_df["transaction_count"])
aml_df["sent_log"] = np.log1p(aml_df["total_sent"])

In [ ]:
aml_df["txn_percentile"] = aml_df["txn_log"].rank(pct=True)
aml_df["sent_percentile"] = aml_df["sent_log"].rank(pct=True)

In [ ]:
aml_df.head(10)

,sender_bank,sender_account,transaction_count,total_sent,unique_receivers,first_txn,last_txn,laundering_count,laundering_rate,total_received,...,active_days,txn_percentile,fan_out_percentile,fan_in_percentile,risk_score,risk_category,sent_percentile,risk_driver,txn_log,sent_log
0,1,800042CB0,134,1.275401e+06,13,2022-09-01 00:13:00,2022-09-10 15:20:00,0,0.000000,40018.84,...,10,0.997579,0.996796,0.346126,78.194866,Medium,0.877853,High Activity,4.905275,14.058772
1,1,800054E60,222,2.781327e+07,13,2022-09-01 02:47:00,2022-09-10 18:55:00,0,0.000000,975938.43,...,10,0.999860,0.996796,0.584992,83.260857,High,0.971398,High Activity,5.407172,17.141024
2,1,8000555D0,128,1.596319e+08,9,2022-09-01 00:20:00,2022-09-10 22:14:00,0,0.000000,3158455.22,...,10,0.997034,0.984577,0.584992,83.121299,High,0.989052,High Activity,4.859812,18.888381
3,1,800056160,89,2.106860e+05,11,2022-09-01 00:02:00,2022-09-10 17:20:00,1,0.011236,6227.20,...,10,0.988884,0.993173,0.987852,85.800198,Critical,0.761985,Laundering,4.499810,12.258129
4,1,800056370,131,5.500107e+07,9,2022-09-01 00:04:00,2022-09-10 23:42:00,0,0.000000,81579.64,...,10,0.997288,0.984577,0.584992,82.994055,High,0.979978,High Activity,4.882802,17.822863
5,1,800056A50,107,1.833335e+07,13,2022-09-01 02:03:00,2022-09-10 17:57:00,0,0.000000,49590.67,...,10,0.994073,0.996796,0.346126,79.363545,Medium,0.963946,High Activity,4.682131,16.724233
6,1,800056CB0,114,1.404377e+06,12,2022-09-01 00:02:00,2022-09-10 07:56:00,0,0.000000,40942.76,...,10,0.995312,0.995318,0.346126,78.148995,Medium,0.882549,High Activity,4.744932,14.155105
7,1,800057020,210,3.497914e+06,15,2022-09-01 01:46:00,2022-09-10 23:21:00,0,0.000000,142707.98,...,10,0.999826,0.998359,0.346126,78.957337,Medium,0.920837,High Activity,5.351858,15.067678
8,1,800057620,45,2.006922e+06,3,2022-09-01 00:03:00,2022-09-09 22:16:00,0,0.000000,98948.11,...,9,0.947933,0.848399,0.584992,76.647951,Medium,0.899028,Normal,3.828641,14.512113
9,1,800057A10,123,4.913023e+07,6,2022-09-01 00:00:00,2022-09-10 22:29:00,0,0.000000,49099916.06,...,10,0.996508,0.956161,0.346126,78.653569,Medium,0.978658,High Activity,4.820282,17.709985


In [ ]:
aml_df.to_csv("aml_table_final.csv", index=False)

In [ ]:
from google.colab import files
files.download("aml_table_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
aml_df.to_parquet("aml_table_final.parquet", index=False)

In [ ]:
from google.colab import files
files.download("aml_table_final.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd


df = pd.read_csv("aml_table_final.csv")


df_sample = df.sample(n=1000, random_state=42)


df_sample.to_csv("sample_transactions.csv", index=False)

In [ ]:
from google.colab import files
files.download("sample_transactions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>